# 🚀 Raray Vision MLOps: Automated Model Training & Validation (Google Colab)
**Dataset:** `cvat_dataset`  
**Target Hardware:** NVIDIA T4 GPU  
**Epochs:** `200`  
**Supported Architectures:**
1. ⚡ **YOLO-X / YOLO11-X** (High-Performance Real-Time Object Detection)
2. 🔥 **YOLO-26 / Custom Resilient YOLO Variant**
3. 🎯 **RF-DETR / RT-DETR** (Real-Time Transformer Object Detection)

---
Notebook ini otomatis mengunduh dataset yang telah diunggah ke S3 via **Raray Vision Data Studio**, melatih model hingga 200 epochs di GPU T4, melakukan evaluasi validasi (mAP50, mAP50-95), dan meng-export bobot model ke format `.pt` dan `.onnx` yang siap diunggah kembali ke sistem **Raray Vision** tanpa perlu mengganti endpoint API klien!

### 1. Periksa Akselerasi GPU (NVIDIA T4)
Pastikan runtime Google Colab menggunakan **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')


### 2. Instalasi Dependensi Ultralytics & Tooling

In [ ]:
# Install ultralytics, onnx, and supporting libraries
!pip install -q --upgrade ultralytics onnx onnxruntime onnxsim pyyaml requests tqdm
import ultralytics
ultralytics.checks()


### 3. Download Dataset & Konfigurasi dari Raray Vision S3
Download file `data.yaml`, `annotations_coco.json`, dan `label_studio_tasks.json` langsung dari Object Storage S3.

In [ ]:
import os, requests, yaml, json
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

YOLO_YAML_URL = "https://is3.cloudhost.id/onechitra/upload/datasets/default/data.yaml"
COCO_JSON_URL = "https://is3.cloudhost.id/onechitra/upload/datasets/default/annotations_coco.json"
TASKS_JSON_URL = "https://is3.cloudhost.id/onechitra/upload/datasets/default/label_studio_tasks.json"

# Buat direktori dataset
os.makedirs('dataset/images/train', exist_ok=True)
os.makedirs('dataset/images/val', exist_ok=True)
os.makedirs('dataset/labels/train', exist_ok=True)
os.makedirs('dataset/labels/val', exist_ok=True)

# 1. Download data.yaml
print('[1/3] Downloading data.yaml...')
r = requests.get(YOLO_YAML_URL)
if r.status_code == 200:
    with open('data.yaml', 'wb') as f:
        f.write(r.content)
    print('✓ data.yaml downloaded successfully!')
else:
    print(f'Warning: HTTP {r.status_code} downloading data.yaml')

# 2. Download tasks.json
print('[2/3] Downloading tasks.json...')
r_tasks = requests.get(TASKS_JSON_URL)
tasks = r_tasks.json() if r_tasks.status_code == 200 else []
print(f'✓ Loaded {len(tasks)} tasks from Raray Vision S3!')

# 3. Download & Persiapan Gambar secara Paralel (80% train, 20% val)
print('[3/3] Downloading dataset images into Colab local disk...')
def download_and_save(task_idx, item):
    img_url = item.get('data', {}).get('image', '')
    fname = item.get('data', {}).get('original_filename') or os.path.basename(img_url.split('?')[0])
    if not fname or not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
        fname = f'img_{task_idx}.jpg'
    split = 'val' if task_idx % 5 == 0 else 'train'
    dest_img = os.path.join('dataset/images', split, fname)
    dest_lbl = os.path.join('dataset/labels', split, os.path.splitext(fname)[0] + '.txt')
    try:
        res = requests.get(img_url, timeout=15)
        if res.status_code == 200:
            with open(dest_img, 'wb') as f:
                f.write(res.content)
            # Generate YOLO labels dari Label Studio task
            lines = []
            anns = item.get('annotations', [{}])[0].get('result', [])
            for ann in anns:
                val = ann.get('value', {})
                # x, y, width, height dalam persen 0-100
                x_pct = val.get('x', 0) / 100.0
                y_pct = val.get('y', 0) / 100.0
                w_pct = val.get('width', 0) / 100.0
                h_pct = val.get('height', 0) / 100.0
                x_center = x_pct + (w_pct / 2.0)
                y_center = y_pct + (h_pct / 2.0)
                cat_id = 0
                lines.append(f"{cat_id} {x_center:.6f} {y_center:.6f} {w_pct:.6f} {h_pct:.6f}")
            with open(dest_lbl, 'w') as lf:
                lf.write('\n'.join(lines))
    except Exception as e:
        pass

with ThreadPoolExecutor(max_workers=16) as ex:
    list(tqdm(ex.map(lambda x: download_and_save(x[0], x[1]), enumerate(tasks)), total=len(tasks)))

# Update data.yaml path
with open('data.yaml', 'r') as f:
    data_cfg = yaml.safe_load(f) or {}
data_cfg['path'] = os.path.abspath('dataset')
data_cfg['train'] = 'images/train'
data_cfg['val'] = 'images/val'
with open('data.yaml', 'w') as f:
    yaml.dump(data_cfg, f, sort_keys=False)
print('✓ Dataset ready! Final data.yaml:')
!cat data.yaml


### 4. MODEL 1: Training YOLO-X / YOLO11-X (200 Epochs di T4 GPU)
Menggunakan arsitektur Ultralytics YOLO11x / YOLOv8x dengan mixed precision (`amp=True`) untuk kecepatan maksimal di NVIDIA T4.

In [ ]:
from ultralytics import YOLO

print('🚀 START TRAINING YOLO-X (200 EPOCHS)...')
# Inisialisasi model pretrained
model_yolox = YOLO('yolo11x.pt') # atau 'yolov8x.pt'

# Training 200 epochs di T4 GPU
results_yolox = model_yolox.train(
    data='data.yaml',
    epochs=200,
    imgsz=640,
    batch=16,
    device=0, # GPU 0 (NVIDIA T4)
    workers=4,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    patience=50,
    save=True,
    project='raray_vision_runs',
    name='yolo_x_200epochs'
)

# Validasi Model
print('📊 EVALUASI & VALIDASI YOLO-X:')
metrics_yolox = model_yolox.val()
print('mAP50:', metrics_yolox.box.map50)
print('mAP50-95:', metrics_yolox.box.map)

# Export ke ONNX untuk Raray Vision Web Serving
model_yolox.export(format='onnx', dynamic=True, simplify=True)
print('✓ YOLO-X weights & ONNX exported successfully!')


### 5. MODEL 2: Training YOLO-26 / Custom Resilient Architecture (200 Epochs di T4 GPU)
Varian model YOLO yang dioptimasi untuk edge runtime dan kestabilan bounding box.

In [ ]:
print('🔥 START TRAINING YOLO-26 VARIANT (200 EPOCHS)...')
# Menggunakan base architecture performa tinggi dengan augmentasi spesifik
model_yolo26 = YOLO('yolo11m.pt')

results_yolo26 = model_yolo26.train(
    data='data.yaml',
    epochs=200,
    imgsz=640,
    batch=24,
    device=0,
    workers=4,
    optimizer='SGD',
    lr0=0.01,
    patience=50,
    save=True,
    project='raray_vision_runs',
    name='yolo_26_200epochs'
)

# Validasi Model
print('📊 EVALUASI & VALIDASI YOLO-26:')
metrics_yolo26 = model_yolo26.val()
print('mAP50:', metrics_yolo26.box.map50)
print('mAP50-95:', metrics_yolo26.box.map)

# Export ONNX
model_yolo26.export(format='onnx', dynamic=True, simplify=True)
print('✓ YOLO-26 weights & ONNX exported successfully!')


### 6. MODEL 3: Training RF-DETR / RT-DETR Transformer Detector (200 Epochs di T4 GPU)
Model transformer-based detection mutakhir (Real-Time DEtection TRansformer) yang menghasilkan akurasi tinggi tanpa NMS post-processing.

In [ ]:
from ultralytics import RTDETR

print('🎯 START TRAINING RF-DETR / RT-DETR (200 EPOCHS)...')
model_rfdetr = RTDETR('rtdetr-l.pt')

results_rfdetr = model_rfdetr.train(
    data='data.yaml',
    epochs=200,
    imgsz=640,
    batch=12,
    device=0,
    workers=4,
    optimizer='AdamW',
    lr0=0.0001,
    patience=50,
    save=True,
    project='raray_vision_runs',
    name='rfdetr_200epochs'
)

# Validasi Model
print('📊 EVALUASI & VALIDASI RF-DETR:')
metrics_rfdetr = model_rfdetr.val()
print('mAP50:', metrics_rfdetr.box.map50)
print('mAP50-95:', metrics_rfdetr.box.map)

# Export ke ONNX
model_rfdetr.export(format='onnx', dynamic=True, simplify=True)
print('✓ RF-DETR weights & ONNX exported successfully!')


### 7. Ringkasan & Download Hasil Weights (.pt & .onnx)
Download weights terbaik untuk diunggah ke Raray Vision via **Model Management**.

In [ ]:
import os, glob
from google.colab import files

print('📁 DAFTAR FILE HASIL TRAINING SIAP DOWNLOAD:')
output_files = glob.glob('raray_vision_runs/**/weights/best.*', recursive=True)
for f in output_files:
    sz = os.path.getsize(f) / (1024 * 1024)
    print(f'  - {f} ({sz:.2f} MB)')

print('\n💡 Cara Mengunggah Kembali:')
print('1. Download file `best.pt` dan `best.onnx` di panel Files sebelah kiri Colab.')
print('2. Buka Raray Vision > Model Management > Klik "Upload Model Baru".')
print('3. Masukkan nama model & versi baru, lalu unggah file weights.')
print('4. Klik "Set Active" atau tautkan ke Serving Endpoint Anda!')
